# DB7 Exercise B: fixed EMG fallback and five-fold subject-disjoint evaluation
Original CNN plus training-fitted EMG shrinkage LDA. Frozen fallback: CNN confidence <0.70 and EMG confidence >0.80. Two separately reported protocols: 22 within-subject validation runs, and five subject-disjoint train/validation/test folds. All repetitions are used in the latter protocol. No thresholds are tuned on new validation or test scores.


In [ ]:
import os, gc, time, json, warnings, random, re, shutil
from pathlib import Path
from copy import deepcopy
from math import gcd

import numpy as np
import pandas as pd
from scipy import io
from scipy.signal import (
    resample_poly, butter, sosfiltfilt, iirnotch, filtfilt
)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score,
    roc_curve,
)

try:
    from thop import profile as thop_profile
    HAS_THOP = True
except ImportError:
    HAS_THOP = False

warnings.filterwarnings('default')
import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [ ]:
def _find_kaggle_input() -> Path:
    base = Path('/kaggle/input')
    if not base.exists():
        return Path('/kaggle/input/ninapro-db7/Dataset')

    def _has_subjects(p):
        return p.is_dir() and any(
            c.is_dir() and c.name.lower().startswith('subject_')
            for c in p.iterdir()
        )

    def _search(root, depth=0):
        if depth > 5:
            return None
        if _has_subjects(root):
            return root
        try:
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    result = _search(child, depth + 1)
                    if result is not None:
                        return result
        except PermissionError:
            pass
        return None

    result = _search(base)
    return result if result else Path('/kaggle/input/ninapro-db7/Dataset')


class Config:
    KAGGLE_INPUT = _find_kaggle_input()
    KAGGLE_WORKING = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()/'eda_working'
    EXERCISE_IDS = (1,)
    GESTURE_MIN, GESTURE_MAX, N_CLASSES = 1, 17, 17
    SUBJECTS = list(range(1,23))
    RUN_SUBJECTS = SUBJECTS.copy()
    INTACT_SUBJECTS = list(range(1,21))
    AMPUTEE_SUBJECTS = [21,22]
    SEED = 42
    MODEL_SEED = 42
    EMG_FS, ACC_FS, TARGET_FS = 2000, 128, 2000
    EMG_KEY, ACC_KEY, LBL_KEY = 'emg', 'acc', 'restimulus'
    N_EMG_CH = 12
    USE_ACC = True
    REPS_PER_GESTURE, TRAIN_REPS, VAL_REPS, TEST_REPS = 6, 4, 1, 1
    BANDPASS_LOW_HZ, BANDPASS_HIGH_HZ, FILTER_ORDER = 20., 450., 4
    NOTCH_HZ, NOTCH_Q = 50., 30.
    WIN_MS, STEP_MS, TRIM_MS = 400, 100, 100
    WIN_SAMPLES, STEP_SAMPLES, TRIM_SAMPLES = 800, 200, 200
    DROPOUT, LR, WEIGHT_DECAY, GRAD_CLIP = .15, 3e-4, 1e-4, 5.
    MIN_EPOCHS, MAX_EPOCHS, PATIENCE, BATCH_SIZE = 20, 150, 15, 128
    NUM_WORKERS = 0
    USE_ADAMW, AUGMENT_TRAIN, EVALUATE_TEST, REFIT_ON_TRAIN_PLUS_VAL = False, False, False, False
    LABEL_SMOOTHING = 0.
    EMG_GAIN_STD, EMG_NOISE_STD = .1, .01
    RUN_CNN = True
    RAW_CASES_PER_SUBJECT = 2
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    AUTOMATION = {}

DIAG_RAW_AUDIT = []
Config.KAGGLE_WORKING.mkdir(parents=True,exist_ok=True)
print('Exercise B only: E1 labels 1–17. Test repetitions are excluded from EDA.')


## Strict source loading and repetition-local preprocessing
The test split is identified by metadata and never filtered or evaluated. Equal exported EMG/ACC row counts are required; this does not independently verify timestamps.

In [ ]:
class RawEMGACCPreprocessor:
    """
    Load RAW EMG, RAW accelerometer and labels from one NinaPro file.

    No filtering, no ACC resampling, no label masking and no windowing happen here.
    Keeping each source file separate preserves the exact temporal mapping needed to
    extract a matching ACC interval for every EMG repetition.
    """

    @staticmethod
    def _find_key(data, candidates):
        for candidate in candidates:
            for key in data:
                if key.lower() == candidate.lower():
                    return key
        return None

    @staticmethod
    def _time_major(array, expected_channels=None, name='signal'):
        array = np.asarray(array)

        if array.ndim == 1:
            array = array[:, None]
        if array.ndim != 2:
            raise RuntimeError(
                f'{name}: expected 2-D array, got shape {array.shape}.'
            )

        if expected_channels is not None:
            if array.shape[1] == expected_channels:
                return array
            if array.shape[0] == expected_channels:
                return array.T
            raise RuntimeError(
                f'{name}: neither dimension matches expected '
                f'{expected_channels} channels: {array.shape}.'
            )

        # For ACC the time dimension should be much larger than channel count.
        if array.shape[0] < array.shape[1] and array.shape[0] <= 128:
            array = array.T

        return array

    def apply(self, mat_path: Path):
        data = io.loadmat(str(mat_path))

        emg_key = self._find_key(data, [Config.EMG_KEY])
        acc_key = self._find_key(data, [Config.ACC_KEY])
        lbl_key = self._find_key(
            data,
            [Config.LBL_KEY, 'stimulus', 'label', 'labels'],
        )

        if emg_key is None:
            raise KeyError(f'No EMG key in {mat_path.name}.')
        if acc_key is None:
            raise KeyError(
                f'No ACC key in {mat_path.name}; EMG+ACC experiment requires accelerometer data.'
            )
        if lbl_key is None:
            raise KeyError(f'No label key in {mat_path.name}.')

        emg = self._time_major(
            data[emg_key],
            expected_channels=Config.N_EMG_CH,
            name=f'{mat_path.name} EMG',
        ).astype(np.float32)

        acc = self._time_major(
            data[acc_key],
            expected_channels=None,
            name=f'{mat_path.name} ACC',
        ).astype(np.float32)

        labels = np.asarray(
            data[lbl_key]
        ).reshape(-1).astype(np.int32)

        # restimulus is aligned with the EMG time base.
        if len(emg) != len(labels):
            raise ValueError('EMG/label length mismatch; truncation is disabled')
        if lbl_key.lower() != 'restimulus':
            raise ValueError('Refined restimulus labels are required')
        n = len(emg)
        emg = emg[:n]
        labels = labels[:n]

        if len(acc) < 2:
            raise RuntimeError(
                f'{mat_path.name}: ACC has only {len(acc)} samples.'
            )

        # Read-only structural audit. No change to preprocessing or labels.
        rep_key = self._find_key(data, ['rerepetition'])
        native = np.asarray(data[rep_key]).reshape(-1) if rep_key else None
        self.last_native_repetition = (native[:n].astype(np.int32)
            if native is not None and len(native) >= n else None)
        self.last_file_path = str(mat_path)
        unique_labels = np.unique(labels).astype(int).tolist()
        run_audit = []
        edges = np.r_[0, np.flatnonzero(np.diff(labels) != 0)+1, len(labels)]
        for start, end in zip(edges[:-1], edges[1:]):
            gesture = int(labels[start])
            if not Config.GESTURE_MIN <= gesture <= Config.GESTURE_MAX:
                continue
            ids = (np.unique(self.last_native_repetition[start:end]).astype(int).tolist()
                   if self.last_native_repetition is not None else [])
            run_audit.append(dict(gesture=gesture,start=int(start),end=int(end),
                duration_seconds=float((end-start)/Config.EMG_FS),native_rerepetition_ids=ids))
        ratio = len(acc)/float(len(emg))
        DIAG_RAW_AUDIT.append(dict(file=str(mat_path),emg_shape=list(emg.shape),
            acc_shape=list(acc.shape),original_emg_shape=list(np.shape(data[emg_key])),
            original_label_length=int(np.size(data[lbl_key])),label_key=lbl_key,
            labels_present=unique_labels,rerepetition_key=rep_key,
            rerepetition_length=int(len(native)) if native is not None else None,
            sample_count_ratio_acc_to_emg=ratio,
            implied_acc_rate_if_shared_duration=ratio*Config.EMG_FS,
            timing_interpretation=('Equal sample counts: may already be aligned; timestamps unverified.'
                if len(acc)==len(emg) else 'Unequal lengths: length-ratio mapping assumes shared start/end times; unverified.'),
            selected_gesture_runs=run_audit))
        return emg, acc, labels


class RepetitionEMGFilter:
    """Zero-phase EMG filtering applied to one already-assigned repetition only."""

    def __init__(self):
        nyquist = Config.EMG_FS / 2.0
        self.sos = butter(
            Config.FILTER_ORDER,
            [
                Config.BANDPASS_LOW_HZ / nyquist,
                Config.BANDPASS_HIGH_HZ / nyquist,
            ],
            btype='bandpass',
            output='sos',
        )
        self.b_notch, self.a_notch = iirnotch(
            Config.NOTCH_HZ / nyquist,
            Config.NOTCH_Q,
        )

    def apply(self, repetition_emg):
        repetition_emg = np.asarray(
            repetition_emg,
            dtype=np.float32,
        )

        if repetition_emg.ndim != 2:
            raise ValueError(
                f'Expected (time,channels), got {repetition_emg.shape}.'
            )
        if len(repetition_emg) < 64:
            raise RuntimeError(
                f'EMG repetition unexpectedly short: {len(repetition_emg)} samples.'
            )

        filtered = sosfiltfilt(
            self.sos,
            repetition_emg,
            axis=0,
        )
        filtered = filtfilt(
            self.b_notch,
            self.a_notch,
            filtered,
            axis=0,
        )
        return filtered.astype(np.float32)


class RepetitionACCResampler:
    """
    Map an EMG repetition interval to the corresponding interval of the RAW ACC
    recording from the SAME source file, then resample only that ACC slice.

    This is leakage-safe because resample_poly never sees ACC samples belonging
    to another train/validation/test repetition.
    """

    @staticmethod
    def extract_matching_interval(
        file_acc: np.ndarray,
        file_emg_length: int,
        emg_start: int,
        emg_end: int,
    ) -> np.ndarray:
        if not (0 <= emg_start < emg_end <= file_emg_length):
            raise ValueError(
                f'Invalid EMG interval [{emg_start}, {emg_end}) '
                f'for file length {file_emg_length}.'
            )

        ratio = len(file_acc) / float(file_emg_length)

        # ceil(start) and ceil(end) keep selected ACC timestamps inside the
        # EMG repetition's temporal interval rather than borrowing a sample
        # from the preceding repetition/rest segment.
        acc_start = int(np.ceil(emg_start * ratio))
        acc_end = int(np.ceil(emg_end * ratio))

        acc_start = max(0, min(acc_start, len(file_acc) - 1))
        acc_end = max(acc_start + 1, min(acc_end, len(file_acc)))

        acc_rep = file_acc[acc_start:acc_end].copy()
        if len(acc_rep) < 2:
            raise RuntimeError(
                f'Mapped ACC repetition is too short: {len(acc_rep)} samples.'
            )

        return acc_rep

    @staticmethod
    def resample_to_emg_length(
        acc_rep: np.ndarray,
        target_length: int,
    ) -> np.ndarray:
        source_length = len(acc_rep)
        divisor = gcd(source_length, target_length)
        up = target_length // divisor
        down = source_length // divisor

        resampled = resample_poly(
            acc_rep,
            up,
            down,
            axis=0,
        ).astype(np.float32)

        # scipy normally gives the exact target length for this reduced ratio.
        # Handle any implementation-level one-sample discrepancy without using
        # data outside this repetition.
        if len(resampled) > target_length:
            resampled = resampled[:target_length]
        elif len(resampled) < target_length:
            pad_count = target_length - len(resampled)
            pad = np.repeat(
                resampled[-1:, :],
                pad_count,
                axis=0,
            )
            resampled = np.vstack([resampled, pad])

        if len(resampled) != target_length:
            raise RuntimeError(
                f'ACC resampling length mismatch: '
                f'{len(resampled)} != {target_length}.'
            )

        return resampled.astype(np.float32)


print('Raw EMG+ACC loader, repetition-local EMG filter and ACC resampler defined.')

In [ ]:
class SubjectLoader:
    """
    Disk-safe subject loader.

    Only ONE subject is held in memory at a time. No raw signal cache and no
    overlapping-window cache is written to /kaggle/working.
    """

    def __init__(self):
        self.preprocessor = RawEMGACCPreprocessor()
        self.rep_filter = RepetitionEMGFilter()
        self.acc_resampler = RepetitionACCResampler()

    def _find_subject_dir(self, sid: int) -> Path:
        candidates = [
            Config.KAGGLE_INPUT / f'Subject_{sid}',
            Config.KAGGLE_INPUT / f'subject_{sid}',
            Config.KAGGLE_INPUT / f'S{sid}',
            Config.KAGGLE_INPUT / f's{sid}',
        ]
        for path in candidates:
            if path.is_dir():
                return path

        for path in sorted(Config.KAGGLE_INPUT.iterdir()):
            if path.is_dir() and path.name.lower().endswith(str(sid)):
                return path

        raise FileNotFoundError(
            f'Cannot find subject {sid} under {Config.KAGGLE_INPUT}.'
        )

    @staticmethod
    def _is_exercise_file(path: Path, exercise_id: int) -> bool:
        name = path.stem.upper()
        return bool(
            re.search(rf'(^|_)E{exercise_id}(_|$)', name)
        )

    def _selected_files(self, sid: int):
        subject_dir = self._find_subject_dir(sid)
        all_mat_files = (
            sorted(subject_dir.glob('*.mat'))
            or sorted(subject_dir.rglob('*.mat'))
        )

        selected = []
        for exercise_id in Config.EXERCISE_IDS:
            matches = [
                path
                for path in all_mat_files
                if self._is_exercise_file(path, exercise_id)
            ]
            if not matches:
                raise FileNotFoundError(
                    f'S{sid:02d}: missing E{exercise_id}. '
                    f'Available: {[p.name for p in all_mat_files[:15]]}'
                )
            selected.extend(matches)

        return selected

    def _load_raw_parts_from_source(self, sid: int):
        """
        Read E1/E2 directly from the read-only Kaggle input and keep them only
        for the current subject.
        """
        parts = []
        acc_channel_counts = set()

        for index, mat_file in enumerate(self._selected_files(sid)):
            emg, acc, labels = self.preprocessor.apply(mat_file)

            if emg.shape[1] != Config.N_EMG_CH:
                raise RuntimeError(
                    f'S{sid:02d}, {mat_file.name}: expected '
                    f'{Config.N_EMG_CH} EMG channels, got {emg.shape[1]}.'
                )

            if acc.shape[1] <= 0:
                raise RuntimeError(
                    f'S{sid:02d}, {mat_file.name}: ACC is required.'
                )

            acc_channel_counts.add(int(acc.shape[1]))

            parts.append({
                'file_index': int(index),
                'file_name': mat_file.name,
                'emg': emg.astype(np.float32, copy=False),
                'acc': acc.astype(np.float32, copy=False),
                'labels': labels.astype(np.int32, copy=False),
                'native_repetition': self.preprocessor.last_native_repetition,
                'source_path': self.preprocessor.last_file_path,
            })

        if len(acc_channel_counts) != 1:
            raise RuntimeError(
                f'S{sid:02d}: inconsistent ACC channel counts across E1/E2: '
                f'{sorted(acc_channel_counts)}.'
            )

        return parts, int(next(iter(acc_channel_counts)))

    @staticmethod
    def _constant_label_runs(labels: np.ndarray):
        if len(labels) == 0:
            return

        boundaries = np.flatnonzero(
            np.diff(labels) != 0
        ) + 1
        starts = np.r_[0, boundaries]
        ends = np.r_[boundaries, len(labels)]

        for start, end in zip(starts, ends):
            yield (
                int(start),
                int(end),
                int(labels[start]),
            )

    @staticmethod
    def _repetition_id(sid, gesture, repetition_index):
        return (
            int(sid),
            int(gesture),
            int(repetition_index + 1),
        )

    @staticmethod
    def _split_repetition_indices(sid, gesture):
        rng = np.random.default_rng(
            Config.SEED
            + 1009 * int(sid)
            + 9176 * int(gesture)
        )
        perm = rng.permutation(
            Config.REPS_PER_GESTURE
        ).tolist()

        test_idx = sorted(
            perm[:Config.TEST_REPS]
        )
        val_idx = sorted(
            perm[
                Config.TEST_REPS:
                Config.TEST_REPS + Config.VAL_REPS
            ]
        )
        train_idx = sorted(
            perm[
                Config.TEST_REPS
                + Config.VAL_REPS:
            ]
        )

        if (
            len(train_idx) != Config.TRAIN_REPS
            or len(val_idx) != Config.VAL_REPS
            or len(test_idx) != Config.TEST_REPS
        ):
            raise RuntimeError('Unexpected repetition split size.')

        return train_idx, val_idx, test_idx

    def _collect_repetitions(self, parts):
        runs_by_gesture = {
            gesture: []
            for gesture in range(
                Config.GESTURE_MIN,
                Config.GESTURE_MAX + 1,
            )
        }

        for part_index, part in enumerate(parts):
            for (
                run_start,
                run_end,
                gesture,
            ) in self._constant_label_runs(part['labels']):
                if gesture in runs_by_gesture:
                    runs_by_gesture[gesture].append({
                        'part_index': int(part_index),
                        'start': int(run_start),
                        'end': int(run_end),
                    })

        return runs_by_gesture



In [ ]:
class ChannelNormalizer:
    """Per-channel mean/std. Fit only on the current subject's training pool."""

    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, X: np.ndarray):
        if len(X) == 0:
            raise ValueError('Cannot fit ChannelNormalizer on an empty array.')
        self.mean = X.mean(axis=(0, 2), keepdims=True)
        self.std = X.std(axis=(0, 2), keepdims=True) + 1e-8
        return self

    def transform(self, X: np.ndarray) -> np.ndarray:
        if self.mean is None or self.std is None:
            raise RuntimeError('ChannelNormalizer must be fitted before transform().')
        return ((X - self.mean) / self.std).astype(np.float32)

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        return self.fit(X).transform(X)


print('Training-only ChannelNormalizer defined.')

In [ ]:
class EMGWindowDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X.astype(np.float32, copy=False))
        self.y = torch.from_numpy(y.astype(np.int64, copy=False))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
import torch
import torch.nn as nn


class ParallelMultiKernelBlock(nn.Module):
    """
    True multi-kernel block: parallel branches with different kernel sizes
    (default 3, 5, 7) processed at the SAME depth and concatenated along the
    channel dimension, then merged with a 1x1 conv. This captures multi-scale
    temporal patterns simultaneously (unlike a sequential 7->5->3 design,
    which only changes kernel size across depth, not within one stage).
    """

    def __init__(
        self,
        in_ch,
        out_ch,
        kernels=(3, 5, 7),
        pool=True,
        dropout=0.1,
    ):
        super().__init__()

        for k in kernels:
            assert k % 2 == 1, (
                f"kernel size {k} must be odd so that "
                f"padding=kernel//2 gives symmetric 'same' padding"
            )

        branch_sizes = self._split_channels(out_ch, len(kernels))

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_ch, b_ch, k, padding=k // 2, bias=False),
                nn.BatchNorm1d(b_ch),
                nn.ReLU(inplace=True),
                nn.Conv1d(b_ch, b_ch, k, padding=k // 2, bias=False),
                nn.BatchNorm1d(b_ch),
                nn.ReLU(inplace=True),
            )
            for k, b_ch in zip(kernels, branch_sizes)
        ])

        self.merge = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
        )

        # Dropout1d zeroes whole feature channels (not individual elements) —
        # correct form of regularization for conv feature maps.
        self.drop = nn.Dropout1d(dropout) if dropout > 0 else nn.Identity()
        self.pool = nn.MaxPool1d(2) if pool else nn.Identity()

    @staticmethod
    def _split_channels(total, n):
        # Splits `total` channels as evenly as possible across `n` branches
        # so the concatenated output is exactly `out_ch` (e.g. 64 -> [22,21,21]).
        base, rem = divmod(total, n)
        return [base + 1 if i < rem else base for i in range(n)]

    def forward(self, x):
        x = torch.cat([branch(x) for branch in self.branches], dim=1)
        x = self.merge(x)
        x = self.drop(x)
        return self.pool(x)


class ChannelAttentionBlock(nn.Module):
    """
    Squeeze-and-Excitation style CHANNEL attention. Learns which learned feature channels
    matter most; it does NOT attend across time
    steps. Named explicitly so it isn't confused with temporal attention.
    """

    def __init__(self, in_ch, reduction=4):
        super().__init__()
        reduced = max(in_ch // reduction, 1)

        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(in_ch, reduced, 1),
            nn.ReLU(inplace=True),
            nn.Conv1d(reduced, in_ch, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.attention(x)


class TemporalAttentionPool(nn.Module):
    """
    Learned attention-weighted pooling over the time axis, replacing plain
    global average pooling. A 1x1 conv scores every time step, softmax turns
    scores into weights, and the weighted sum replaces a uniform mean — so
    the model can down-weight uninformative (e.g. resting) time steps instead
    of averaging them in blindly.
    """

    def __init__(self, in_ch):
        super().__init__()
        self.score = nn.Conv1d(in_ch, 1, kernel_size=1)

    def forward(self, x):
        # x: (batch, channels, time)
        weights = torch.softmax(self.score(x), dim=-1)   # (batch, 1, time)
        return (x * weights).sum(dim=-1)                 # (batch, channels)


class OriginalMultiKernelAttention1DCNN(nn.Module):
    def __init__(
        self,
        n_channels,
        n_classes,
        dropout=0.15,
    ):
        super().__init__()
        self.n_channels = int(n_channels)

        self.stage1 = ParallelMultiKernelBlock(
            n_channels, 64, kernels=(3, 5, 7), pool=True, dropout=dropout
        )
        self.attn1 = ChannelAttentionBlock(64)

        self.stage2 = ParallelMultiKernelBlock(
            64, 128, kernels=(3, 5, 7), pool=True, dropout=dropout
        )
        self.attn2 = ChannelAttentionBlock(128)

        self.stage3 = ParallelMultiKernelBlock(
            128, 256, kernels=(3, 5, 7), pool=False, dropout=dropout
        )
        self.attn3 = ChannelAttentionBlock(256)

        self.temporal_pool = TemporalAttentionPool(256)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.stage1(x)
        x = self.attn1(x)
        x = self.stage2(x)
        x = self.attn2(x)
        x = self.stage3(x)
        x = self.attn3(x)
        x = self.temporal_pool(x)
        return self.head(x)

    def count_params(self):
        return sum(
            parameter.numel()
            for parameter in self.parameters()
            if parameter.requires_grad
        )



import torch.nn.functional as F



"""Paper-inspired 1D amplitude/velocity ablation; not a T-EKIM implementation."""

def derivative_statistics(X, fs=2000.0, n_emg=12, batch_size=128):
    """Fit only on normalized TRAIN windows; omit padded boundary positions.

    X has shape (windows, original channels, time). Derivatives are computed
    independently inside each window, so no repetition boundary is crossed.
    Overlapping training windows contribute repeated samples, as in the existing
    channel normalizer. No validation or test array is accepted by the runner.
    """
    if X.ndim != 3 or len(X) == 0 or X.shape[1] < n_emg or X.shape[2] < 2:
        raise ValueError('Expected nonempty (windows, channels, time>=2).')
    if fs <= 0:
        raise ValueError('Sampling rate must be positive.')
    total = np.zeros(n_emg, dtype=np.float64)
    total_sq = np.zeros(n_emg, dtype=np.float64)
    count = 0
    for start in range(0, len(X), batch_size):
        chunk = X[start:start+batch_size, :n_emg].astype(np.float64)
        derivative = np.diff(chunk, axis=-1) * fs
        if not np.isfinite(derivative).all():
            raise FloatingPointError('Nonfinite training derivative.')
        total += derivative.sum(axis=(0, 2))
        total_sq += np.square(derivative).sum(axis=(0, 2))
        count += derivative.shape[0] * derivative.shape[2]
    mean = total / count
    std = np.sqrt(np.maximum(total_sq/count - mean**2, 0))
    return mean.astype(np.float32), np.maximum(std, 1e-6).astype(np.float32)


class MultiKernelAttention1DCNN(nn.Module):
    """Original multi-kernel CNN with optional 12 standardized EMG derivatives.

    Input remains B x 48 x 800 (12 EMG, then 36 ACC); the candidate appends
    12 derivatives internally, giving B x 60 x 800. The original convolution
    blocks, kernels, widths, attention, pooling and head are unchanged.
    Constructor metadata + state_dict + original normalizer suffice to reload.
    """
    def __init__(self, n_channels, n_classes, dropout=0.15,
                 variant='baseline', modality='both', n_emg=12, fs=2000.0):
        super().__init__()
        if variant not in ('baseline', 'amplitude_velocity'):
            raise ValueError('Use baseline or amplitude_velocity.')
        if modality != 'both':
            raise ValueError('This controlled comparison keeps both EMG and ACC.')
        if n_channels <= n_emg:
            raise ValueError('Expected EMG followed by ACC channels.')
        self.n_channels = int(n_channels)
        self.n_emg = int(n_emg)
        self.fs = float(fs)
        self.variant, self.modality = variant, modality
        self.register_buffer('derivative_mean', torch.zeros(1, n_emg, 1))
        self.register_buffer('derivative_std', torch.ones(1, n_emg, 1))
        self.register_buffer('derivative_fitted', torch.tensor(False))
        self.network = OriginalMultiKernelAttention1DCNN(
            n_channels + (n_emg if variant == 'amplitude_velocity' else 0),
            n_classes, dropout)

    def fit_preprocessor(self, X_train):
        if self.variant == 'amplitude_velocity':
            mean, std = derivative_statistics(X_train, self.fs, self.n_emg)
            with torch.no_grad():
                self.derivative_mean.copy_(torch.as_tensor(mean).reshape(1, -1, 1))
                self.derivative_std.copy_(torch.as_tensor(std).reshape(1, -1, 1))
                self.derivative_fitted.fill_(True)
        return self

    def prepare_input(self, x, mask_derivative=False):
        if x.ndim != 3 or x.shape[1] != self.n_channels or x.shape[-1] < 2:
            raise ValueError('Expected (batch, original channels, time>=2).')
        if self.variant == 'baseline':
            return x
        if not self.derivative_fitted.item():
            raise RuntimeError('Fit derivative statistics on training data or load a fitted checkpoint first.')
        derivative = (x[:, :self.n_emg, 1:] - x[:, :self.n_emg, :-1]) * self.fs
        derivative = (derivative-self.derivative_mean) / self.derivative_std
        # No preceding sample exists at t=0. Pad with zero in standardized units.
        derivative = F.pad(derivative, (1, 0), value=0.0)
        if mask_derivative:
            derivative = torch.zeros_like(derivative)
        return torch.cat([x, derivative], dim=1)

    def forward(self, x, mask_derivative=False):
        return self.network(self.prepare_input(x, mask_derivative))

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def save_derivative_health(model, X_train, X_val, directory):
    """Read-only summary of the fitted transform; never refits on validation."""
    if model.variant != 'amplitude_velocity':
        return
    means=model.derivative_mean.detach().cpu().numpy().reshape(-1)
    stds=model.derivative_std.detach().cpu().numpy().reshape(-1)
    rows=[]
    for split,X in [('train',X_train),('validation',X_val)]:
        ids=np.linspace(0,len(X)-1,min(128,len(X)),dtype=int)
        d=np.diff(X[ids,:model.n_emg].astype(np.float64),axis=-1)*model.fs
        z=(d-means[None,:,None])/stds[None,:,None]
        for ch in range(model.n_emg):
            rows.append(dict(split=split,emg_channel=ch,sampled_windows=len(ids),
                fitted_derivative_mean=float(means[ch]),fitted_derivative_std=float(stds[ch]),
                standardized_derivative_mean=float(z[:,ch].mean()),
                standardized_derivative_std=float(z[:,ch].std()),
                extreme_fraction_abs_gt_10=float((np.abs(z[:,ch])>10).mean())))
    pd.DataFrame(rows).to_csv(directory/'derivative_health.csv',index=False)
    np.savez_compressed(directory/'derivative_normalizer.npz',mean=means,std=stds,fs=model.fs)


In [ ]:
class Trainer:
    def __init__(self, model, save_path: Path, n_classes: int):
        self.model = model.to(Config.DEVICE)
        self.save_path = save_path
        self.n_classes = n_classes
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_acc': [],
            'val_acc': [],
        }
        self.best_epoch = 0
        self.best_val_loss = float('inf')
        self.train_wall = 0.0

    def _run_epoch(self, loader, optimizer=None, criterion=None):
        training = optimizer is not None
        self.model.train(training)

        total_loss, correct, total = 0.0, 0, 0
        context = torch.enable_grad() if training else torch.no_grad()

        with context:
            for X, y in loader:
                X = X.to(Config.DEVICE, non_blocking=True)
                y = y.to(Config.DEVICE, non_blocking=True)

                if training and Config.AUGMENT_TRAIN:
                    X = X.clone()
                    emg = X[:, :Config.N_EMG_CH]
                    gain = torch.exp(Config.EMG_GAIN_STD * torch.randn(
                        emg.shape[0], emg.shape[1], 1, device=emg.device))
                    X[:, :Config.N_EMG_CH] = (emg * gain
                        + Config.EMG_NOISE_STD * torch.randn_like(emg))
                logits = self.model(X)
                loss = criterion(logits, y)

                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(),
                        Config.GRAD_CLIP,
                    )
                    optimizer.step()

                total_loss += loss.item() * len(y)
                correct += (logits.argmax(1) == y).sum().item()
                total += len(y)

        return total_loss / max(total, 1), correct / max(total, 1)

    def fit(self, train_loader, val_loader):
        optimizer = (torch.optim.AdamW if Config.USE_ADAMW else Adam)(
            self.model.parameters(),
            lr=Config.LR,
            weight_decay=Config.WEIGHT_DECAY,
        )
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=Config.MAX_EPOCHS,
        )

        patience_count = 0
        start_wall = time.perf_counter()

        for epoch in range(1, Config.MAX_EPOCHS + 1):
            tr_loss, tr_acc = self._run_epoch(
                train_loader, optimizer, criterion
            )
            vl_loss, vl_acc = self._run_epoch(
                val_loader, criterion=criterion
            )
            scheduler.step()

            self.history['train_loss'].append(tr_loss)
            self.history['val_loss'].append(vl_loss)
            self.history['train_acc'].append(tr_acc)
            self.history['val_acc'].append(vl_acc)

            if vl_loss < self.best_val_loss:
                self.best_val_loss = float(vl_loss)
                self.best_epoch = int(epoch)
                patience_count = 0
                torch.save(self.model.state_dict(), self.save_path)
            else:
                patience_count += 1

            if epoch == 1 or epoch % 10 == 0:
                print(
                    f'  Epoch {epoch:3d} | '
                    f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} | '
                    f'val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}'
                )

            if epoch >= Config.MIN_EPOCHS and patience_count >= Config.PATIENCE:
                print(
                    f'  Early stop at epoch {epoch} '
                    f'(patience={Config.PATIENCE})'
                )
                break

        self.train_wall = time.perf_counter() - start_wall
        print(
            f'  Selection training: {self.train_wall:.1f} s | '
            f'best epoch={self.best_epoch} | '
            f'best val_loss={self.best_val_loss:.4f}'
        )
        return self

    def fit_fixed_epochs(self, train_loader, epochs: int):
        """Fresh final model training on train+validation after epoch selection."""
        optimizer = (torch.optim.AdamW if Config.USE_ADAMW else Adam)(
            self.model.parameters(),
            lr=Config.LR,
            weight_decay=Config.WEIGHT_DECAY,
        )
        criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)
        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=Config.MAX_EPOCHS  # preserve the selection LR trajectory,
        )

        start_wall = time.perf_counter()
        for epoch in range(1, int(epochs) + 1):
            loss, acc = self._run_epoch(
                train_loader, optimizer, criterion
            )
            scheduler.step()

            if epoch == 1 or epoch % 10 == 0 or epoch == int(epochs):
                print(
                    f'  Refit epoch {epoch:3d}/{int(epochs)} | '
                    f'loss={loss:.4f} | acc={acc:.4f}'
                )

        self.train_wall = time.perf_counter() - start_wall
        torch.save(self.model.state_dict(), self.save_path)
        print(f'  Refit training: {self.train_wall:.1f} s')
        return self


print('Trainer defined.')

In [ ]:
import sys, platform, traceback, zipfile
from datetime import datetime, timezone
from sklearn.metrics import classification_report, balanced_accuracy_score
from scipy.signal import welch
def json_write(path, obj):
    def convert(x):
        if isinstance(x, np.ndarray):
            return x.tolist()
        if isinstance(x, np.generic):
            return x.item()
        return str(x)
    Path(path).write_text(json.dumps(obj, indent=2, default=convert), encoding='utf-8')

def save_figure(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)

def probability_metrics(y, p):
    """Uncalibrated confidence diagnostics; multiclass Brier is a sum over classes."""
    pred = p.argmax(1)
    confidence = p.max(1)
    correct = pred == y
    bins = np.minimum((confidence * 10).astype(int), 9)
    calibration = []
    ece = 0.0
    for b in range(10):
        mask = bins == b
        n = int(mask.sum())
        acc = float(correct[mask].mean()) if n else None
        conf = float(confidence[mask].mean()) if n else None
        calibration.append(dict(bin=b, lower=b/10, upper=(b+1)/10,
                                count=n, accuracy=acc, confidence=conf))
        if n:
            ece += n / len(y) * abs(acc - conf)
    targets = np.eye(p.shape[1])[y]
    result = dict(
        accuracy=float(correct.mean()),
        balanced_accuracy=float(balanced_accuracy_score(y, pred)),
        f1_macro=float(f1_score(y, pred, labels=np.arange(p.shape[1]),
                              average='macro', zero_division=0)),
        f1_weighted=float(f1_score(y, pred, average='weighted', zero_division=0)),
        nll=float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean()),
        brier_multiclass=float(((p - targets)**2).sum(1).mean()),
        ece_10_bins=float(ece),
        high_confidence_error_fraction=float(((confidence >= .9) & ~correct).mean()),
        n_windows=int(len(y)))
    return result, calibration

def predict_arrays(model, X, mask=None):
    """Order-preserving prediction, optional zero-at-training-mean stress test."""
    model.eval()
    out = []
    with torch.no_grad():
        for start in range(0, len(X), Config.BATCH_SIZE):
            x = torch.as_tensor(X[start:start+Config.BATCH_SIZE], device=Config.DEVICE)
            if mask is not None:
                x = x.clone()
                if mask == 'emg':
                    x[:, :Config.N_EMG_CH] = 0
                elif mask == 'acc':
                    x[:, Config.N_EMG_CH:] = 0
                else:
                    x[:, int(mask)] = 0
            p = torch.softmax(model(x), dim=1)
            if not torch.isfinite(p).all():
                raise FloatingPointError('Non-finite predictions')
            out.append(p.cpu().numpy())
    return np.concatenate(out)

class DiagnosticTrainer(Trainer):
    """Preserves Trainer.fit selection logic; records each epoch without extra forwards."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.epoch_rows = []
        self.last_training = {}

    def _run_epoch(self, loader, optimizer=None, criterion=None):
        training = optimizer is not None
        self.model.train(training)
        total_loss, correct, total = 0.0, 0, 0
        norms, pred_all, y_all = [], [], []
        tick = time.perf_counter()
        with torch.set_grad_enabled(training):
            for X, y in loader:
                X, y = X.to(Config.DEVICE), y.to(Config.DEVICE)
                if training and Config.AUGMENT_TRAIN:
                    X = X.clone()
                    emg = X[:, :Config.N_EMG_CH]
                    gain = torch.exp(Config.EMG_GAIN_STD * torch.randn(
                        emg.shape[0], emg.shape[1], 1, device=emg.device))
                    X[:, :Config.N_EMG_CH] = emg * gain + Config.EMG_NOISE_STD * torch.randn_like(emg)
                logits = self.model(X)
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Non-finite loss; inspect signal_health.csv')
                if training:
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
                    if not torch.isfinite(norm):
                        raise FloatingPointError('Non-finite gradient norm')
                    norms.append(float(norm.item()))
                    optimizer.step()
                pred = logits.argmax(1)
                total_loss += loss.item() * len(y)
                correct += (pred == y).sum().item()
                total += len(y)
                pred_all.extend(pred.detach().cpu().tolist())
                y_all.extend(y.cpu().tolist())
        values = dict(loss=total_loss/total, accuracy=correct/total,
                      f1_macro=float(f1_score(y_all, pred_all, labels=range(self.n_classes),
                                             average='macro', zero_division=0)),
                      seconds=time.perf_counter()-tick)
        labels=np.asarray(y_all); guesses=np.asarray(pred_all)
        if not hasattr(self,'class_epoch_rows'): self.class_epoch_rows=[]
        for c in range(self.n_classes):
            selected=labels==c
            self.class_epoch_rows.append(dict(epoch=len(self.epoch_rows)+1,split='train_online_dropout' if training else 'validation',
                gesture=c+Config.GESTURE_MIN,windows=int(selected.sum()),recall=float((guesses[selected]==c).mean())))
        pd.DataFrame(self.class_epoch_rows).to_csv(self.save_path.parent/'class_learning_history.csv',index=False)
        if training:
            self.last_training = {'train_'+k:v for k,v in values.items()}
            self.last_training.update(lr=optimizer.param_groups[0]['lr'],
                gradient_norm_mean=float(np.mean(norms)), gradient_norm_max=float(np.max(norms)),
                gradient_clipped_fraction=float(np.mean(np.array(norms)>Config.GRAD_CLIP)))
        else:
            row = dict(epoch=len(self.epoch_rows)+1, **self.last_training,
                       **{'val_'+k:v for k,v in values.items()})
            self.epoch_rows.append(row)
            # Persist after every epoch: useful even if Kaggle interrupts later.
            pd.DataFrame(self.epoch_rows).to_csv(self.save_path.parent/'history.csv', index=False)
        return values['loss'], values['accuracy']

def signal_diagnostics(data, normalizer, directory):
    """Train/validation only. Samples windows to bound temporary memory."""
    rows = []
    for split in ('train', 'val'):
        X = data['X_'+split]
        ids = np.linspace(0, len(X)-1, min(len(X), 128), dtype=int)
        sample = X[ids]
        for ch in range(X.shape[1]):
            x = sample[:, ch, :].astype(np.float64)
            finite = np.isfinite(x)
            good = x[finite]
            mu = float(normalizer.mean[0, ch, 0])
            sd = float(normalizer.std[0, ch, 0])
            rows.append(dict(split=split, channel=ch,
                modality='emg' if ch < Config.N_EMG_CH else 'acc',
                sampled_windows=len(ids), nonfinite_fraction=float(1-finite.mean()),
                mean=float(good.mean()) if good.size else None,
                std=float(good.std()) if good.size else None,
                normalized_mean=float((good.mean()-mu)/sd) if good.size else None,
                normalized_std=float(good.std()/sd) if good.size else None,
                normalized_extreme_fraction=float((np.abs((good-mu)/sd)>10).mean()) if good.size else None,
                constant_within_window_fraction=float((np.ptp(x, axis=1)<1e-12).mean())))
    stats = pd.DataFrame(rows)
    stats.to_csv(directory/'signal_health.csv', index=False)
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))
    for split in ('train', 'val'):
        part = stats[stats.split == split]
        axes[0].plot(part.channel, part.normalized_mean, label=split)
        axes[1].plot(part.channel, part.normalized_std, label=split)
    for ax in axes:
        ax.axvline(11.5, color='gray', linestyle='--')
        ax.legend()
        ax.set_xlabel('Input channel: EMG 0–11; ACC 12–47')
    axes[0].set_ylabel('Mean in training SD units')
    axes[1].set_ylabel('SD / training SD')
    save_figure(fig, directory/'channel_shift.png')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    spectral = []
    for split in ('train', 'val'):
        X = data['X_'+split]
        ids = np.linspace(0, len(X)-1, min(24, len(X)), dtype=int)
        for ax, (name, channel_slice) in zip(axes, [('emg', slice(0,12)), ('acc', slice(12,None))]):
            f, p = welch(X[ids, channel_slice], fs=Config.TARGET_FS,
                         nperseg=min(512, X.shape[-1]), axis=-1)
            power = p.mean(axis=(0,1))
            ax.semilogy(f, power+1e-20, label=split)
            spectral.extend(dict(split=split, modality=name, frequency_hz=float(a), power=float(b))
                            for a,b in zip(f,power))
            ax.set_xlim(0, 500 if name=='emg' else 100)
            ax.set_title(name.upper()+' after existing preprocessing')
            ax.set_xlabel('Hz'); ax.legend()
    save_figure(fig, directory/'signal_spectrum.png')
    pd.DataFrame(spectral).to_csv(directory/'signal_spectrum.csv', index=False)
    return stats

def prediction_report(model, X, y, metadata, directory, split):
    p = predict_arrays(model, X)
    metrics, calibration = probability_metrics(y, p)
    pred, conf = p.argmax(1), p.max(1)
    frame = pd.DataFrame(metadata).copy()
    assert len(frame) == len(y)
    frame['true_class'] = y
    frame['predicted_class'] = pred
    frame['predicted_gesture'] = pred + Config.GESTURE_MIN
    frame['correct'] = pred == y
    frame['confidence'] = conf
    frame['true_class_probability'] = p[np.arange(len(y)), y]
    top3 = np.argsort(-p, axis=1)[:, :3]
    for rank in range(3):
        frame[f'top{rank+1}_gesture'] = top3[:,rank]+Config.GESTURE_MIN
    frame.to_csv(directory/f'{split}_predictions.csv', index=False)
    np.savez_compressed(directory/f'{split}_probabilities.npz', y_true=y, probabilities=p)
    frame[~frame.correct].sort_values('confidence', ascending=False).head(100).to_csv(
        directory/f'{split}_confident_errors.csv', index=False)
    cm = confusion_matrix(y,pred,labels=np.arange(Config.N_CLASSES))
    class_names = [f'G{i}' for i in range(Config.GESTURE_MIN,Config.GESTURE_MAX+1)]
    pd.DataFrame(cm,index=class_names,columns=class_names).to_csv(directory/f'{split}_confusion_counts.csv')
    report = classification_report(y,pred,labels=np.arange(Config.N_CLASSES),
        target_names=class_names,output_dict=True,zero_division=0)
    pd.DataFrame(report).T.to_csv(directory/f'{split}_classification_report.csv')
    if split == 'validation':
        cmn = cm / np.maximum(cm.sum(1,keepdims=True),1)
        fig, ax = plt.subplots(figsize=(11,9))
        sns.heatmap(cmn,ax=ax,vmin=0,vmax=1,cmap='Blues',annot=True,fmt='.2f',
                    xticklabels=class_names,yticklabels=class_names)
        ax.set(xlabel='Predicted gesture',ylabel='True gesture',title='Validation recall by gesture')
        save_figure(fig,directory/'validation_confusion.png')
        pairs = [(class_names[i],class_names[j],int(cm[i,j]),float(cmn[i,j]))
                 for i in range(len(cm)) for j in range(len(cm)) if i!=j and cm[i,j]]
        pd.DataFrame(sorted(pairs,key=lambda z:-z[2]),columns=['true','predicted','count','fraction_of_true_class']).to_csv(
            directory/'confusion_pairs.csv',index=False)
        cal = pd.DataFrame(calibration)
        cal.to_csv(directory/'calibration.csv',index=False)
        fig, axes = plt.subplots(1,2,figsize=(10,4))
        active = cal[cal['count']>0]
        axes[0].plot(active.confidence,active.accuracy,'o-')
        axes[0].plot([0,1],[0,1],'--',color='gray')
        axes[0].set(xlabel='Mean confidence',ylabel='Observed accuracy',title='Reliability (uncalibrated)')
        for correct,label in [(True,'Correct'),(False,'Wrong')]:
            axes[1].hist(conf[(pred==y)==correct],bins=np.linspace(0,1,11),alpha=.6,label=label)
        axes[1].set(xlabel='Confidence',ylabel='Window count'); axes[1].legend()
        save_figure(fig,directory/'confidence.png')
        fig, axes = plt.subplots(2,1,figsize=(13,5),sharex=True)
        axes[0].plot(frame.gesture.to_numpy(),'.',label='True')
        axes[0].plot(frame.predicted_gesture.to_numpy(),'.',alpha=.5,label='Predicted')
        axes[0].legend(); axes[0].set_ylabel('Gesture ID')
        axes[1].plot(conf); axes[1].set(xlabel='Window index (class/repetition order; not continuous time)',ylabel='Confidence')
        save_figure(fig,directory/'validation_prediction_sequence.png')
    # These are within-repetition window summaries, not additional independent trials.
    frame.groupby(['file_name','gesture','run_index'],dropna=False).agg(
        windows=('correct','size'),window_accuracy=('correct','mean'),
        mean_confidence=('confidence','mean')).reset_index().to_csv(directory/f'{split}_repetition_summary.csv',index=False)
    json_write(directory/f'{split}_metrics.json',metrics)
    return metrics,p

def learning_report(trainer, directory):
    h = pd.DataFrame(trainer.epoch_rows)
    h.to_csv(directory/'history.csv',index=False)
    fig, axes = plt.subplots(2,2,figsize=(12,8))
    for split in ('train','val'):
        axes[0,0].plot(h.epoch,h[split+'_loss'],label=split)
        axes[0,1].plot(h.epoch,h[split+'_accuracy'],label=split)
    axes[0,0].set_ylabel('Cross entropy'); axes[0,1].set_ylabel('Accuracy')
    axes[1,0].plot(h.epoch,h.lr); axes[1,0].set_ylabel('Learning rate')
    axes[1,1].plot(h.epoch,h.gradient_norm_mean,label='Mean before clipping')
    axes[1,1].plot(h.epoch,h.gradient_norm_max,label='Max before clipping')
    axes[1,1].axhline(Config.GRAD_CLIP,color='gray',linestyle='--')
    axes[1,1].set_ylabel('Gradient norm')
    for ax in axes.flat:
        ax.axvline(trainer.best_epoch,color='red',linestyle=':',label='Selected epoch')
        ax.set_xlabel('Epoch'); ax.legend(fontsize=8)
    fig.suptitle('Online training uses dropout; checkpoint gap is measured separately in eval mode')
    save_figure(fig,directory/'learning_curves.png')



## EDA and model diagnostics
Feature scaling, LDA and PCA use training data only. Raw exports are limited to two worst validation gestures per subject, each with its four training repetitions as controls. Signal summaries and features cover every development gesture.

In [ ]:
"""Exercise B EDA implementation embedded into the self-contained notebook."""
from scipy.signal import welch
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from threadpoolctl import threadpool_limits


def feature_matrix(X):
    # Physical filtered EMG and aligned ACC; no per-window standardization.
    e=X[:,:12].astype(np.float64); a=X[:,12:].astype(np.float64)
    diff=np.diff(e,axis=-1)
    spec=np.abs(np.fft.rfft(e,axis=-1))**2
    freq=np.fft.rfftfreq(e.shape[-1],1/Config.EMG_FS)
    total=spec.sum(-1)+1e-30
    meanfreq=(spec*freq).sum(-1)/total
    medianfreq=freq[np.argmax(np.cumsum(spec,axis=-1)>=total[:,:,None]/2,axis=-1)]
    blocks=[np.sqrt(np.mean(e**2,-1)),np.mean(abs(e),-1),np.mean(abs(diff),-1),
            np.mean(e[:,:,:-1]*e[:,:,1:]<0,-1),meanfreq,medianfreq,
            a.mean(-1),a.std(-1),np.mean(abs(np.diff(a,axis=-1)),-1)]
    names=[f'emg_{name}_{c+1:02}' for name in ['rms','mav','wl_mean','zc_fraction','mean_frequency','median_frequency'] for c in range(12)]
    names += [f'acc_{name}_{c+1:02}' for name in ['mean','std','mean_abs_diff'] for c in range(36)]
    features=np.concatenate(blocks,axis=1).astype(np.float32)
    if not np.isfinite(features).all(): raise FloatingPointError('Nonfinite features')
    return features,names


def repetition_health(raw,filtered,acc,meta):
    rows=[]
    for modality,signal in [('raw_emg',raw),('filtered_emg',filtered),('acc',acc)]:
        freq,power=welch(signal,fs=Config.EMG_FS,nperseg=min(1024,len(signal)),axis=0)
        for ch in range(signal.shape[1]):
            x=signal[:,ch].astype(np.float64); pp=power[:,ch];den=max(float(pp.sum()),1e-30)
            row=dict(**meta,modality=modality,channel=ch+1,mean=float(x.mean()),std=float(x.std()),
                rms=float(np.sqrt(np.mean(x*x))),mav=float(abs(x).mean()),peak_abs=float(abs(x).max()),
                repeated_adjacent_fraction=float(np.mean(np.diff(x)==0)),
                extrema_fraction=float(np.mean((x==x.min())|(x==x.max()))),
                line_50_fraction=float(pp[(freq>=45)&(freq<=55)].sum()/den),
                below_20_fraction=float(pp[freq<20].sum()/den),
                above_450_fraction=float(pp[freq>450].sum()/den))
            rows.append(row)
    return rows


def build_eda_subject(sid,root):
    loader=SubjectLoader(); parts,nacc=loader._load_raw_parts_from_source(sid)
    if len(parts)!=1 or nacc!=36: raise ValueError('Exercise B expects exactly one E1 file and 36 ACC channels')
    part=parts[0]
    if set(np.unique(part['labels'])) != set(range(18)): raise ValueError('E1 must contain rest 0 and movements 1–17')
    if part['native_repetition'] is None: raise ValueError('rerepetition is required for an auditable split')
    if len(part['emg'])!=len(part['acc']): raise ValueError('Unequal EMG/ACC row counts require an explicit timing review before this EDA')
    if not np.isfinite(part['emg']).all() or not np.isfinite(part['acc']).all(): raise ValueError('Nonfinite source data')
    runs=loader._collect_repetitions(parts)
    bank={};allmeta=[];health=[];assign={};xs={'train':[],'val':[]};ys={'train':[],'val':[]};wm={'train':[],'val':[]}
    for g,reps in runs.items():
        if len(reps)!=6: raise ValueError(f'S{sid}: label {g} has {len(reps)} runs, expected 6')
        train,val,test=loader._split_repetition_indices(sid,g)
        assign[g]={'train':[i+1 for i in train],'val':[i+1 for i in val],'test':[i+1 for i in test]}
        native_seen=set()
        for ri,rec in enumerate(reps):
            start,end=rec['start'],rec['end'];native=np.unique(part['native_repetition'][start:end])
            if len(native)!=1 or int(native[0]) not in range(1,7) or int(native[0]) in native_seen:
                raise ValueError(f'Native repetition mismatch: S{sid}, G{g}, run {ri+1}')
            native_seen.add(int(native[0]))
            split='train' if ri in train else 'val' if ri in val else 'test'
            meta=dict(subject=sid,gesture=g,run_index=ri+1,native_repetition=int(native[0]),split=split,
                file_name=part['file_name'],run_start_emg_sample=start,run_end_emg_sample=end,duration_seconds=(end-start)/2000)
            allmeta.append(meta)
            if split=='test': continue  # No test filtering, features, waveform export or predictions.
            raw=part['emg'][start:end].copy(); acc=part['acc'][start:end].copy(); filt=loader.rep_filter.apply(raw)
            # Adjacent rest only; stop before any other active repetition.
            context_start=start;context_end=end;labels=part['labels']
            while context_start>max(0,start-1000) and labels[context_start-1]==0:context_start-=1
            while context_end<min(len(labels),end+1000) and labels[context_end]==0:context_end+=1
            bank[(g,ri+1)]=dict(meta=meta,raw=raw,filtered=filt,acc=acc,
                context_emg=part['emg'][context_start:context_end].copy(),
                context_labels=labels[context_start:context_end].copy(),context_start=context_start)
            health.extend(repetition_health(raw,filt,acc,meta))
            signal=np.concatenate([filt,acc],axis=1)
            starts=list(range(Config.TRIM_SAMPLES,len(signal)-Config.TRIM_SAMPLES-Config.WIN_SAMPLES+1,Config.STEP_SAMPLES))
            if not starts: raise ValueError('No windows after boundary trimming')
            for w in starts:
                xs[split].append(signal[w:w+Config.WIN_SAMPLES].T.copy());ys[split].append(g-1)
                wm[split].append(dict(subject=sid,split=split,file_name=part['file_name'],source_path=part['source_path'],
                    gesture=g,run_index=ri+1,run_start_emg_sample=start,run_end_emg_sample=end,
                    window_start_emg_sample=start+w,window_end_emg_sample=start+w+Config.WIN_SAMPLES,
                    native_rerepetition_ids=str(int(native[0]))))
    data={}
    for split in ['train','val']:
        data['X_'+split]=np.stack(xs[split]);data['y_'+split]=np.asarray(ys[split],dtype=np.int64);data['meta_'+split]=wm[split]
        assert set(data['y_'+split])==set(range(17))
        pd.DataFrame(wm[split]).to_csv(root/f'{split}_window_provenance.csv',index=False)
    pd.DataFrame(allmeta).to_csv(root/'repetition_inventory.csv',index=False)
    pd.DataFrame(health).to_csv(root/'repetition_signal_health.csv',index=False)
    json_write(root/'assignments.json',assign)
    json_write(root/'integrity_checks.json',dict(passed=True,exercise='B',file_exercise=1,labels=list(range(1,18)),
        repetitions_per_gesture=6,test_policy='metadata only; excluded from EDA and model evaluation',
        timing='Equal exported row counts required; acquisition timestamps not independently verified'))
    del parts,part,xs;gc.collect()
    return data,bank


def feature_eda(sid,data,root):
    features={};columns=None
    for split in ['train','val']:
        blocks=[]
        for k in range(0,len(data['X_'+split]),128):
            f,columns=feature_matrix(data['X_'+split][k:k+128]);blocks.append(f)
        features[split]=np.concatenate(blocks)
        frame=pd.DataFrame(data['meta_'+split]);frame=pd.concat([frame,pd.DataFrame(features[split],columns=columns)],axis=1)
        frame.to_csv(root/f'{split}_window_features.csv',index=False)
    feature_dir=root/'feature_models';feature_dir.mkdir()
    summaries=[];preds={}
    for modality,selection in [('emg',slice(0,72)),('acc',slice(72,None)),('both',slice(None))]:
        # Fixed shrinkage LDA; no tuning on validation, no window-random cross validation.
        pipe=make_pipeline(StandardScaler(),LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'))
        with threadpool_limits(limits=2):
            pipe.fit(features['train'][:,selection],data['y_train'])
            p=pipe.predict_proba(features['val'][:,selection])
        metric,_=probability_metrics(data['y_val'],p);preds[modality]=p
        summaries.append(dict(subject=sid,model='shrinkage_lda',modality=modality,**metric))
        np.savez_compressed(feature_dir/f'{modality}_probabilities.npz',y_true=data['y_val'],probabilities=p)
        cm=confusion_matrix(data['y_val'],p.argmax(1),labels=range(17));pd.DataFrame(cm,index=range(1,18),columns=range(1,18)).to_csv(feature_dir/f'{modality}_confusion.csv')
    pd.DataFrame(summaries).to_csv(feature_dir/'summary.csv',index=False)
    scaler=StandardScaler().fit(features['train']);zt=scaler.transform(features['train']);zv=scaler.transform(features['val'])
    pca=PCA(n_components=2,svd_solver='full').fit(zt);pt=pca.transform(zt);pv=pca.transform(zv)
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    for ax,z,split in zip(axes,[pt,pv],['train','val']):
        ax.scatter(z[:,0],z[:,1],c=data['y_'+split],cmap='tab20',s=5);ax.set(title=f'{split}: training-fitted feature PCA',xlabel='PC1',ylabel='PC2')
        pd.concat([pd.DataFrame(data['meta_'+split]),pd.DataFrame(z,columns=['PC1','PC2'])],axis=1).to_csv(root/f'{split}_feature_pca.csv',index=False)
    save_figure(fig,root/'feature_pca.png')
    np.savez_compressed(root/'feature_transform.npz',mean=scaler.mean_,scale=scaler.scale_,pca_components=pca.components_,pca_mean=pca.mean_,explained_variance_ratio=pca.explained_variance_ratio_)
    shifts=[];proto=[];proto_meta=[]
    tm=pd.DataFrame(data['meta_train']);vm=pd.DataFrame(data['meta_val'])
    for (g,rep),ids in tm.groupby(['gesture','run_index']).groups.items():
        proto.append(zt[list(ids)].mean(0));proto_meta.append((g,rep))
    proto=np.asarray(proto)
    for g,ids in vm.groupby('gesture').groups.items():
        valmean=zv[list(ids)].mean(0);trainmean=zt[np.array(data['y_train'])==g-1].mean(0)
        for j,col in enumerate(columns):shifts.append(dict(gesture=g,feature=col,train_mean_z=float(trainmean[j]),validation_mean_z=float(valmean[j]),difference_training_std=float(valmean[j]-trainmean[j])))
    pd.DataFrame(shifts).to_csv(root/'gesture_feature_shift.csv',index=False)
    nearest=[]
    for g,ids in vm.groupby('gesture').groups.items():
        valmean=zv[list(ids)].mean(0);dist=np.sqrt(np.mean((proto-valmean)**2,axis=1))
        for rank,k in enumerate(np.argsort(dist)[:5]):nearest.append(dict(validation_gesture=g,rank=rank+1,training_gesture=proto_meta[k][0],training_repetition=proto_meta[k][1],distance=float(dist[k])))
    pd.DataFrame(nearest).to_csv(root/'nearest_training_repetitions.csv',index=False)
    fig,ax=plt.subplots(figsize=(15,6));shift=pd.DataFrame(shifts).pivot(index='gesture',columns='feature',values='difference_training_std')
    sns.heatmap(shift,ax=ax,cmap='coolwarm',center=0,xticklabels=False);ax.set_title('Gesture-specific validation minus training feature means (training SD units)');save_figure(fig,root/'gesture_feature_shift.png')
    return summaries,preds


def failure_summary(data,train_p,val_p,root):
    rows=[];phases=[]
    for g in range(1,18):
        ti=np.flatnonzero(data['y_train']==g-1);vi=np.flatnonzero(data['y_val']==g-1);p=val_p[vi];correct=p.argmax(1)==g-1
        rank=(p>p[:,g-1,None]).sum(1)+1; streak=best=0
        for hit in correct:streak=0 if hit else streak+1;best=max(best,streak)
        tm=pd.DataFrame(data['meta_train']).iloc[ti];rep=[]
        for r,idx in tm.groupby('run_index').groups.items():rep.append(dict(repetition=int(r),recall=float((train_p[list(idx)].argmax(1)==g-1).mean()),windows=len(idx)))
        rows.append(dict(gesture=g,train_recall=float((train_p[ti].argmax(1)==g-1).mean()),validation_recall=float(correct.mean()),
            validation_windows=len(vi),top3_recall=float((rank<=3).mean()),mean_true_probability=float(p[:,g-1].mean()),
            high_confidence_errors=int(((p.max(1)>=.9)&~correct).sum()),longest_wrong_windows=best,
            longest_wrong_coverage_seconds=0 if not best else .4+.1*(best-1),training_repetition_recalls=json.dumps(rep)))
        for ii,hit in zip(vi,correct):
            m=data['meta_val'][ii];pos=((m['window_start_emg_sample']+m['window_end_emg_sample'])/2-m['run_start_emg_sample'])/(m['run_end_emg_sample']-m['run_start_emg_sample'])
            phases.append(dict(gesture=g,region='first_20pct' if pos<.2 else 'last_20pct' if pos>.8 else 'middle_60pct',correct=bool(hit)))
    pd.DataFrame(rows).to_csv(root/'failure_cases.csv',index=False)
    pd.DataFrame(phases).groupby(['gesture','region']).agg(windows=('correct','size'),accuracy=('correct','mean')).reset_index().to_csv(root/'phase_errors.csv',index=False)
    return rows


def export_case_signals(bank,cases,root):
    selected=sorted(cases,key=lambda r:(r['validation_recall'],-r['high_confidence_errors']))[:Config.RAW_CASES_PER_SUBJECT]
    folder=root/'raw_failure_cases';folder.mkdir()
    for case in selected:
        g=case['gesture'];reps=[b for (gg,r),b in sorted(bank.items()) if gg==g]
        fig,axes=plt.subplots(len(reps),4,figsize=(20,3*len(reps)),squeeze=False)
        for row,b in enumerate(reps):
            meta=b['meta'];name=f'G{g:02}_rep{meta["run_index"]}_{meta["split"]}'
            np.savez_compressed(folder/f'{name}.npz',raw_emg=b['raw'],filtered_emg=b['filtered'],acc=b['acc'],
                context_emg=b['context_emg'],context_labels=b['context_labels'],context_start=b['context_start'],
                emg_fs=2000,export_grid_fs=2000,metadata_json=json.dumps(meta))
            t=np.arange(len(b['raw']))/2000;step=max(1,len(t)//2500)
            axes[row,0].plot(t[::step],b['raw'][::step,0],lw=.5,label='Raw EMG channel 1');axes[row,0].plot(t[::step],b['filtered'][::step,0],lw=.5,alpha=.6,label='Filtered')
            env=np.sqrt(uniform_filter1d(b['filtered'].astype(np.float64)**2,size=100,axis=0,mode='nearest'))
            axes[row,1].plot(t[::step],env[::step],lw=.6);axes[row,1].set_title('50 ms RMS envelopes: all 12 EMG channels')
            axes[row,2].plot(t[::step],b['acc'][::step],lw=.5);axes[row,2].set_title('All 36 ACC axes; exported alignment grid')
            ct=(np.arange(len(b['context_emg']))+b['context_start']-meta['run_start_emg_sample'])/2000
            ce=np.sqrt(uniform_filter1d(np.mean(b['context_emg'].astype(np.float64)**2,axis=1),size=100))
            axes[row,3].plot(ct[::step],ce[::step]);axes[row,3].set_title('Raw RMS and adjacent rest (labels)')
            lab=axes[row,3].twinx();lab.plot(ct[::step],b['context_labels'][::step],color='gray',alpha=.4);lab.set_ylabel('Label')
            axes[row,0].set_title(name)
            for ax in axes[row]:ax.axvline(.1,color='gray',ls=':');ax.axvline(t[-1]-.1,color='gray',ls=':');ax.set_xlabel('Time within repetition (s)')
        save_figure(fig,folder/f'G{g:02}_repetition_comparison.png')


def attribution_and_embeddings(model,Xt,Xv,data,p,root):
    embeddings=[]
    def hook(module,inputs,output):embeddings.append(output.detach().cpu().numpy())
    handle=model.network.head[1].register_forward_hook(hook)
    predict_arrays(model,Xt);et=np.concatenate(embeddings);embeddings.clear();predict_arrays(model,Xv);ev=np.concatenate(embeddings);handle.remove()
    scaler=StandardScaler().fit(et);pca=PCA(n_components=2).fit(scaler.transform(et))
    np.savez_compressed(root/'cnn_embeddings.npz',train=et,validation=ev,y_train=data['y_train'],y_validation=data['y_val'],
        train_pca=pca.transform(scaler.transform(et)),validation_pca=pca.transform(scaler.transform(ev)),scaler_mean=scaler.mean_,scaler_scale=scaler.scale_,pca_components=pca.components_,pca_mean=pca.mean_)
    wrong=np.flatnonzero(p.argmax(1)!=data['y_val']);ids=wrong[np.argsort(-p[wrong].max(1))[:16]]
    if not len(ids):ids=np.arange(min(16,len(Xv)))
    x=torch.tensor(Xv[ids],device=Config.DEVICE,requires_grad=True);y=torch.tensor(data['y_val'][ids],device=Config.DEVICE)
    logits=model(x);gradient=torch.autograd.grad(logits[torch.arange(len(ids),device=Config.DEVICE),y].sum(),x)[0]
    np.savez_compressed(root/'selected_input_gradients.npz',validation_indices=ids,input_normalized=x.detach().cpu().numpy(),
        true_logit_gradient=gradient.detach().cpu().numpy(),y_true=data['y_val'][ids])
    rows=[]
    for ch in range(48):
        scores=predict_arrays(model,Xv[ids],mask=ch)
        rows.append(dict(channel=ch+1,modality='emg' if ch<12 else 'acc',
            mean_true_probability_change=float(np.mean(scores[np.arange(len(ids)),data['y_val'][ids]]-p[ids,data['y_val'][ids]]))))
    pd.DataFrame(rows).to_csv(root/'selected_failure_channel_occlusion.csv',index=False)
    json_write(root/'attribution_notes.json',dict(selected_validation_indices=ids,interpretation='Input gradients and mean-value masking are sensitivity diagnostics, not causal explanations; selection favors confidently wrong windows.'))


def train_eda_cnn(sid,data,normalizer,root):
    folder=root/'cnn_baseline';folder.mkdir();seed=Config.MODEL_SEED+1009*sid
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    Xt=normalizer.transform(data['X_train']);Xv=normalizer.transform(data['X_val'])
    model=MultiKernelAttention1DCNN(48,17,dropout=Config.DROPOUT,variant='baseline',modality='both')
    gen=torch.Generator().manual_seed(seed)
    tl=DataLoader(EMGWindowDataset(Xt,data['y_train']),batch_size=Config.BATCH_SIZE,shuffle=True,generator=gen)
    vl=DataLoader(EMGWindowDataset(Xv,data['y_val']),batch_size=Config.BATCH_SIZE,shuffle=False)
    trainer=DiagnosticTrainer(model,folder/'best_selection.pt',17);trainer.fit(tl,vl);learning_report(trainer,folder)
    model.load_state_dict(torch.load(folder/'best_selection.pt',weights_only=True,map_location=Config.DEVICE))
    tr,tp=prediction_report(model,Xt,data['y_train'],data['meta_train'],folder,'train_eval')
    va,vp=prediction_report(model,Xv,data['y_val'],data['meta_val'],folder,'validation')
    rows=failure_summary(data,tp,vp,folder)
    stress=[]
    for modality in ['emg','acc']:
        m,_=probability_metrics(data['y_val'],predict_arrays(model,Xv,mask=modality));stress.append(dict(masked=modality,accuracy=m['accuracy']))
    pd.DataFrame(stress).to_csv(folder/'modality_mask_stress.csv',index=False)
    attribution_and_embeddings(model,Xt,Xv,data,vp,folder)
    np.savez_compressed(folder/'normalizer.npz',mean=normalizer.mean,std=normalizer.std)
    json_write(folder/'model_config.json',dict(variant='baseline',channels=48,classes=17,exercise='B',labels=list(range(1,18)),seed=seed,selected_epoch=trainer.best_epoch))
    result=dict(subject=sid,model='original_multikernel_cnn',modality='both',selected_epoch=trainer.best_epoch,
        train_accuracy=tr['accuracy'],**va)
    del model,trainer,Xt,Xv,tl,vl;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return result,rows


def eda_preflight(root):
    # Actual tiny fit exercises CPU/GPU, predictions, feature baselines and plotting before real subjects.
    rng=np.random.default_rng(72);X=rng.normal(size=(34,48,800)).astype(np.float32)
    feats,names=feature_matrix(X);assert feats.shape==(34,180) and len(names)==180
    scaler=StandardScaler().fit(feats);assert np.isfinite(scaler.transform(feats)).all()
    model=MultiKernelAttention1DCNN(48,17,variant='baseline').to(Config.DEVICE)
    y=torch.arange(34,device=Config.DEVICE)%17;loss=nn.CrossEntropyLoss()(model(torch.tensor(X,device=Config.DEVICE)),y);loss.backward()
    assert all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters())
    p=predict_arrays(model,X);assert p.shape==(34,17)
    fig,ax=plt.subplots();ax.plot(feats[:,0]);save_figure(fig,root/'synthetic_preflight.png')
    json_write(root/'synthetic_preflight.json',dict(passed=True,synthetic_only=True))
    del model,X;gc.collect()


def run_exercise_b_eda():
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ');root=Config.KAGGLE_WORKING/f'db7_exercise_b_eda_{stamp}';root.mkdir(parents=True)
    manifest={k:getattr(Config,k) for k in dir(Config) if k.isupper()}
    manifest.update(exercise='B',file_exercise=1,labels=list(range(1,18)),test_evaluated=False,
        python=sys.version,torch=torch.__version__,numpy=np.__version__,gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        deterministic_algorithms=torch.are_deterministic_algorithms_enabled())
    json_write(root/'run_manifest.json',manifest);summaries=[];completed=[]
    try:
        assert Config.EXERCISE_IDS==(1,) and Config.GESTURE_MIN==1 and Config.GESTURE_MAX==17
        assert not Config.EVALUATE_TEST and not Config.REFIT_ON_TRAIN_PLUS_VAL
        eda_preflight(root)
        for sid in Config.RUN_SUBJECTS:
            print(f'Exercise B EDA: S{sid:02}',flush=True);sd=root/f'S{sid:02}';sd.mkdir()
            data,bank=build_eda_subject(sid,sd);normalizer=ChannelNormalizer().fit(data['X_train']);signal_diagnostics(data,normalizer,sd)
            fs,probs=feature_eda(sid,data,sd);summaries.extend(fs)
            if Config.RUN_CNN:
                result,cases=train_eda_cnn(sid,data,normalizer,sd);summaries.append(result)
            else:
                # Failure ranking from LDA only when explicitly running the lighter EDA mode.
                y=data['y_val'];cases=[dict(gesture=g,validation_recall=float((probs['both'].argmax(1)[y==g-1]==g-1).mean()),high_confidence_errors=0) for g in range(1,18)]
            export_case_signals(bank,cases,sd)
            pd.DataFrame(summaries).to_csv(root/'model_summary.csv',index=False);completed.append(sid)
            json_write(root/'completion.json',dict(completed_subjects=completed,requested_subjects=Config.RUN_SUBJECTS,success=False))
            json_write(root/'raw_file_audit.json',DIAG_RAW_AUDIT)
            del data,bank,normalizer;gc.collect()
        json_write(root/'completion.json',dict(completed_subjects=completed,requested_subjects=Config.RUN_SUBJECTS,success=True))
        pd.DataFrame([dict(file_exercise=1,exercise='B',file_label=g,model_class=g-1,display_label=f'B{g:02}',movement_name='Confirm against acquisition gesture dictionary') for g in range(1,18)]).to_csv(root/'gesture_dictionary.csv',index=False)
        (root/'READ_ME_RESULTS.md').write_text('Exercise B only: E1 labels 1–17. Test repetitions are metadata-only.\n\nStart with model_summary.csv, then subject failure_cases.csv, phase_errors.csv, gesture_feature_shift.csv and raw_failure_cases/. Class-learning history reports online training with dropout and validation in eval mode; checkpoint train_eval scores are separate. Raw failure archives contain exact train/validation signals and adjacent rest only. No prior mixed-task scores are reused. Feature models use training-fitted transforms and fixed shrinkage LDA. Attribution is sensitivity, not causality. PSD line-power/extrema indicators are heuristics, not proof of mains contamination or clipping. Synthetic preflight is not a dataset result.\n',encoding='utf-8')
    except Exception:
        (root/'FAILURE.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
    finally:
        archive=shutil.make_archive(str(root),'zip',root_dir=root);print('EDA_OUTPUT_ZIP:',archive,flush=True)
    return root


In [ ]:
"""Original CNN + frozen EMG confidence fallback; within-subject and subject-held-out protocols."""
import joblib

Config.GATE_CNN_THRESHOLD=.70
Config.GATE_EMG_THRESHOLD=.80
Config.CV_MAX_EPOCHS=60
Config.CV_MIN_EPOCHS=10
Config.CV_PATIENCE=10
Config.RUN_CROSS_SUBJECT_CV=True
Config.SCOPE='all'

def subject_folds():
    # Fixed random partition independent of outcomes. Each subject is test exactly once.
    order=np.random.default_rng(42).permutation(np.arange(1,23))
    groups=[sorted(map(int,a)) for a in np.array_split(order,5)]
    result=[]
    for i in range(5):
        test=groups[i];val=groups[(i+1)%5];train=sorted(set(range(1,23))-set(test)-set(val))
        assert not (set(train)&set(val) or set(train)&set(test) or set(val)&set(test))
        result.append(dict(fold=i+1,train_subjects=train,validation_subjects=val,test_subjects=test))
    assert sorted(s for f in result for s in f['test_subjects'])==list(range(1,23))
    return result

def cache_subject(sid,folder,meta_folder):
    loader=SubjectLoader();parts,nacc=loader._load_raw_parts_from_source(sid)
    assert len(parts)==1 and nacc==36
    part=parts[0];assert set(np.unique(part['labels']))==set(range(18))
    assert part['native_repetition'] is not None and len(part['emg'])==len(part['acc'])
    assert np.isfinite(part['emg']).all() and np.isfinite(part['acc']).all()
    runs=loader._collect_repetitions(parts);jobs=[];metadata=[];inventory=[]
    for g,reps in runs.items():
        assert len(reps)==6
        train,val,test=loader._split_repetition_indices(sid,g);seen=set()
        for ri,rec in enumerate(reps):
            lo,hi=rec['start'],rec['end'];native=np.unique(part['native_repetition'][lo:hi])
            assert len(native)==1 and int(native[0]) in range(1,7) and int(native[0]) not in seen;seen.add(int(native[0]))
            split='train' if ri in train else 'val' if ri in val else 'test'
            inventory.append(dict(subject=sid,gesture=g,run_index=ri+1,native_repetition=int(native[0]),within_split=split,run_start_emg_sample=lo,run_end_emg_sample=hi))
            starts=range(Config.TRIM_SAMPLES,hi-lo-Config.TRIM_SAMPLES-Config.WIN_SAMPLES+1,Config.STEP_SAMPLES)
            assert len(starts)>0;jobs.append((lo,hi,starts))
            for start in starts:metadata.append(dict(subject=sid,file_name=part['file_name'],gesture=g,run_index=ri+1,native_repetition=int(native[0]),within_split=split,run_start_emg_sample=lo,run_end_emg_sample=hi,window_start_emg_sample=lo+start,window_end_emg_sample=lo+start+Config.WIN_SAMPLES))
    filename=folder/f'S{sid:02}.npy';X=np.lib.format.open_memmap(filename,mode='w+',dtype='float32',shape=(len(metadata),48,800));offset=0
    for lo,hi,starts in jobs:
        emg=loader.rep_filter.apply(part['emg'][lo:hi]);signal=np.concatenate([emg,part['acc'][lo:hi]],axis=1)
        for start in starts:X[offset]=signal[start:start+800].T;offset+=1
    X.flush();assert offset==len(metadata)
    blocks=[]
    for start in range(0,len(X),128):
        f,columns=feature_matrix(X[start:start+128]);blocks.append(f[:,:72])
    features=np.concatenate(blocks);np.save(folder/f'S{sid:02}_features.npy',features)
    meta=pd.DataFrame(metadata);meta.to_csv(meta_folder/f'S{sid:02}_all_window_provenance.csv',index=False)
    pd.DataFrame(inventory).to_csv(meta_folder/f'S{sid:02}_repetition_inventory.csv',index=False)
    del X,parts,part,signal,emg;gc.collect()
    return dict(path=filename,meta=meta,features=features)

class CachedWindows(Dataset):
    def __init__(self,cache,subjects,mean,std,within_split=None):
        self.arrays={s:np.load(cache[s]['path'],mmap_mode='r') for s in subjects};self.mean=mean;self.std=std;self.references=[];frames=[];features=[]
        for s in subjects:
            meta=cache[s]['meta'];ids=np.flatnonzero(meta.within_split.to_numpy()==within_split) if within_split else np.arange(len(meta))
            self.references.extend((s,int(i)) for i in ids);frames.append(meta.iloc[ids]);features.append(cache[s]['features'][ids])
        self.meta=pd.concat(frames,ignore_index=True);self.y=self.meta.gesture.to_numpy(dtype=np.int64)-1;self.features=np.concatenate(features)
    def __len__(self):return len(self.references)
    def __getitem__(self,i):
        s,j=self.references[i];x=((self.arrays[s][j]-self.mean)/self.std).astype(np.float32)
        return torch.from_numpy(x),int(self.y[i])

def fit_normalizer(cache,subjects,within=False):
    if within:
        s=subjects[0];meta=cache[s]['meta'];x=np.load(cache[s]['path'],mmap_mode='r')[meta.within_split.to_numpy()=='train']
        norm=ChannelNormalizer().fit(x);return norm.mean[0],norm.std[0]
    total=np.zeros(48,dtype=np.float64);squares=np.zeros(48,dtype=np.float64);count=0
    for s in subjects:
        x=np.load(cache[s]['path'],mmap_mode='r')
        for k in range(0,len(x),128):
            a=x[k:k+128].astype(np.float64);total+=a.sum(axis=(0,2));squares+=(a*a).sum(axis=(0,2));count+=a.shape[0]*a.shape[2]
    mean=total/count;std=np.sqrt(np.maximum(squares/count-mean**2,0))+1e-8
    return mean.astype(np.float32)[:,None],std.astype(np.float32)[:,None]

def predict_dataset(model,dataset):
    model.eval();result=[]
    with torch.no_grad():
        for x,y in DataLoader(dataset,batch_size=Config.BATCH_SIZE,shuffle=False,num_workers=0):result.append(model(x.to(Config.DEVICE)).softmax(1).cpu().numpy())
    return np.concatenate(result)

def evaluate_gate(model,lda,dataset,path,protocol,unit,split):
    cp=predict_dataset(model,dataset)
    with threadpool_limits(limits=2):ep=lda.predict_proba(dataset.features)
    assert np.array_equal(lda.classes_,np.arange(17))
    switch=(cp.max(1)<Config.GATE_CNN_THRESHOLD)&(ep.max(1)>Config.GATE_EMG_THRESHOLD)
    gate=np.where(switch[:,None],ep,cp);y=dataset.y;base_correct=cp.argmax(1)==y
    path.mkdir(parents=True,exist_ok=True);np.savez_compressed(path/'probabilities.npz',y_true=y,cnn=cp,emg=ep,gate=gate,switch=switch)
    frame=dataset.meta.copy();frame['true_class']=y;frame['cnn_prediction']=cp.argmax(1)+1;frame['emg_prediction']=ep.argmax(1)+1;frame['gate_prediction']=gate.argmax(1)+1;frame['cnn_confidence']=cp.max(1);frame['emg_confidence']=ep.max(1);frame['switch_to_emg']=switch
    frame.to_csv(path/'predictions.csv',index=False);rows=[]
    for arm,p in [('baseline',cp),('emg_lda',ep),('fixed_gate',gate)]:
        metric,calibration=probability_metrics(y,p);correct=p.argmax(1)==y
        row=dict(protocol=protocol,unit=unit,split=split,arm=arm,recovered=int((~base_correct&correct).sum()),harmed=int((base_correct&~correct).sum()),**metric);rows.append(row)
        json_write(path/f'{arm}_metrics.json',row);pd.DataFrame(calibration).to_csv(path/f'{arm}_calibration.csv',index=False)
        pd.DataFrame(confusion_matrix(y,p.argmax(1),labels=range(17)),index=range(1,18),columns=range(1,18)).to_csv(path/f'{arm}_confusion.csv')
    detail=[]
    for (s,g),ids in frame.groupby(['subject','gesture']).groups.items():
        ids=np.array(list(ids));b=base_correct[ids];e=ep[ids].argmax(1)==y[ids];c=gate[ids].argmax(1)==y[ids]
        detail.append(dict(subject=int(s),gesture=int(g),windows=len(ids),baseline_accuracy=float(b.mean()),gate_accuracy=float(c.mean()),recovered=int((~b&c).sum()),harmed=int((b&~c).sum()),emg_rescue_opportunities=int((~b&e).sum()),both_wrong=int((~b&~e).sum())))
    pd.DataFrame(detail).to_csv(path/'gesture_recovery.csv',index=False)
    subject_scores=[]
    for s,ids in frame.groupby('subject').groups.items():
        ids=np.array(list(ids))
        for arm,p in [('baseline',cp),('emg_lda',ep),('fixed_gate',gate)]:
            metric,_=probability_metrics(y[ids],p[ids]);subject_scores.append(dict(protocol=protocol,unit=unit,split=split,subject=int(s),arm=arm,**metric))
    pd.DataFrame(subject_scores).to_csv(path/'subject_metrics.csv',index=False)
    return rows,subject_scores

def fit_unit(cache,train_subjects,val_subjects,test_subjects,path,protocol,unit,smoke=False):
    path.mkdir(parents=True,exist_ok=True);within=protocol=='within_subject';actual_seed=42+1009*(unit if within else 100+unit)
    random.seed(actual_seed);np.random.seed(actual_seed);torch.manual_seed(actual_seed);torch.cuda.manual_seed_all(actual_seed)
    if not within:assert not(set(train_subjects)&set(val_subjects) or set(train_subjects)&set(test_subjects) or set(val_subjects)&set(test_subjects))
    mean,std=fit_normalizer(cache,train_subjects,within);np.savez_compressed(path/'normalizer.npz',mean=mean,std=std)
    train=CachedWindows(cache,train_subjects,mean,std,'train' if within else None);val=CachedWindows(cache,val_subjects,mean,std,'val' if within else None)
    train.meta.to_csv(path/'train_provenance.csv',index=False);val.meta.to_csv(path/'validation_provenance.csv',index=False)
    assert set(train.y)==set(val.y)==set(range(17))
    model=MultiKernelAttention1DCNN(48,17,dropout=Config.DROPOUT,variant='baseline')
    trainer=DiagnosticTrainer(model,path/'best_selection.pt',17);generator=torch.Generator().manual_seed(actual_seed)
    old=(Config.MIN_EPOCHS,Config.MAX_EPOCHS,Config.PATIENCE)
    if not within:Config.MIN_EPOCHS,Config.MAX_EPOCHS,Config.PATIENCE=Config.CV_MIN_EPOCHS,Config.CV_MAX_EPOCHS,Config.CV_PATIENCE
    if smoke:Config.MIN_EPOCHS,Config.MAX_EPOCHS,Config.PATIENCE=1,2,2
    try:trainer.fit(DataLoader(train,batch_size=Config.BATCH_SIZE,shuffle=True,generator=generator,num_workers=0),DataLoader(val,batch_size=Config.BATCH_SIZE,shuffle=False,num_workers=0))
    finally:Config.MIN_EPOCHS,Config.MAX_EPOCHS,Config.PATIENCE=old
    learning_report(trainer,path);model.load_state_dict(torch.load(path/'best_selection.pt',weights_only=True,map_location=Config.DEVICE))
    lda=make_pipeline(StandardScaler(),LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto'))
    with threadpool_limits(limits=2):lda.fit(train.features,train.y)
    joblib.dump(lda,path/'emg_lda.joblib')
    json_write(path/'fit_manifest.json',dict(protocol=protocol,unit=unit,train_subjects=train_subjects,validation_subjects=val_subjects,test_subjects=test_subjects,actual_seed=actual_seed,selected_epoch=trainer.best_epoch,gate_thresholds=[.7,.8],thresholds_fitted=False,cnn_parameters=sum(p.numel() for p in model.parameters()),cross_subject_max_epochs=Config.CV_MAX_EPOCHS,smoke_only=smoke))
    rows,scores=evaluate_gate(model,lda,val,path/'validation',protocol,unit,'validation')
    if not within:
        test=CachedWindows(cache,test_subjects,mean,std);tr,ts=evaluate_gate(model,lda,test,path/'test',protocol,unit,'test');rows+=tr;scores+=ts;del test
    del trainer,model,train,val,lda;gc.collect();torch.cuda.empty_cache()
    return rows,scores

def write_gate_summary(rows,scores,root):
    pd.DataFrame(rows).to_csv(root/'model_summary.csv',index=False);s=pd.DataFrame(scores);s.to_csv(root/'subject_metrics.csv',index=False)
    s.groupby(['protocol','split','arm'])[['accuracy','balanced_accuracy','f1_macro']].mean().to_csv(root/'aggregate_subject_mean.csv')

def run_gate_cv_experiment():
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ');root=Config.KAGGLE_WORKING/f'db7_gate_cv_{stamp}';root.mkdir(parents=True)
    folder=Path('/kaggle/temp')/f'db7_gate_cache_{stamp}';folder.mkdir(parents=True,exist_ok=True);meta_folder=root/'data_provenance';meta_folder.mkdir()
    folds=subject_folds();smoke=Config.SCOPE=='smoke';subjects=[1,2,3] if smoke else list(range(1,23));cache={};rows=[];scores=[];completed=[]
    json_write(root/'fold_assignments.json',folds)
    manifest={k:getattr(Config,k) for k in dir(Config) if k.isupper()};manifest.update(exercise='B',labels=list(range(1,18)),within_subject_test_evaluated=False,cross_subject_test_evaluated=True,protocol_warning='Subject-disjoint CV uses all six repetitions with new subject-level train/validation/test assignments. Previous within-subject reserved repetitions are therefore used in this separate protocol.',threshold_note='Frozen from a previous validation screen; CV is a new protocol, not proof of unbiased post-selection performance.',smoke_only=smoke,torch=torch.__version__,gpu=torch.cuda.get_device_name(0))
    json_write(root/'run_manifest.json',manifest)
    try:
        assert Config.DEVICE.type=='cuda' and Config.EXERCISE_IDS==(1,) and not Config.AUGMENT_TRAIN
        for sid in subjects:
            print(f'Caching all six repetitions S{sid:02}',flush=True);cache[sid]=cache_subject(sid,folder,meta_folder)
            if not smoke or sid==1:
                a,b=fit_unit(cache,[sid],[sid],[],root/'within_subject'/f'S{sid:02}','within_subject',sid,smoke);rows+=a;scores+=b;completed.append(f'within_S{sid:02}');write_gate_summary(rows,scores,root)
        jobs=[dict(fold=0,train_subjects=[1],validation_subjects=[2],test_subjects=[3])] if smoke else folds
        for fold in jobs:
            print('Subject-held-out fold:',fold,flush=True)
            a,b=fit_unit(cache,fold['train_subjects'],fold['validation_subjects'],fold['test_subjects'],root/'cross_subject'/f"fold_{fold['fold']}",'cross_subject',fold['fold'],smoke);rows+=a;scores+=b;completed.append(f"fold_{fold['fold']}");write_gate_summary(rows,scores,root)
            json_write(root/'completion.json',dict(success=False,completed=completed))
        json_write(root/'completion.json',dict(success=True,completed=completed,within_subject_count=1 if smoke else 22,cross_subject_fold_count=1 if smoke else 5,smoke_only=smoke))
        (root/'READ_ME_RESULTS.md').write_text('Compare baseline and fixed_gate within each protocol. Aggregate subject means are in aggregate_subject_mean.csv. Within-subject scores are validation only; cross-subject test predictions are held-out by subject with five fixed folds. Gate uses EMG when CNN max probability <0.70 and EMG max probability >0.80; thresholds are frozen, not fitted. EMG features use only the training split for StandardScaler and shrinkage LDA. Test subjects do not enter normalization, model fitting or checkpoint selection in their fold. Subject groups rotate roles across folds; do not adapt later folds using earlier test results. Previous validation screening influenced this method. Fold 0 and two-epoch runs are smoke checks only. Cached physical signal arrays are excluded from this archive.\n')
    except Exception:
        (root/'FAILURE.txt').write_text(traceback.format_exc());raise
    finally:
        print('EXPERIMENT_OUTPUT_ZIP:',shutil.make_archive(str(root),'zip',root_dir=root),flush=True)
        for p in folder.glob('*.npy'):p.unlink()
        folder.rmdir()
    return root


In [ ]:
EXPERIMENT_RESULTS=run_gate_cv_experiment()
print(EXPERIMENT_RESULTS)
